In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

REPO = "ZhixinYin/qwen25-coder-syntax-repair"
tok = AutoTokenizer.from_pretrained(REPO)
model = AutoModelForCausalLM.from_pretrained(REPO, torch_dtype="auto", device_map="auto").eval()

config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.56k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [ ]:
INSTRUCTION = (
    "The following Python program does not parse. Make the smallest edit that\n"
    "allows it to parse.\n"
    "\n"
    "Preserve the program's behaviour exactly as written, including any logical\n"
    "errors. Do not correct wrong algorithms, wrong operators, wrong variable\n"
    "usage, off-by-one errors, or missing cases. Do not add, remove, or reorder\n"
    "statements beyond what is strictly required to make the code parse. Keep\n"
    "the original variable names. Change whitespace only where the indentation\n"
    "itself is the error.\n"
    "\n"
    "Output only the corrected code \u2014 no explanations, no markdown fences, no\n"
    "leading or trailing text. The result must differ from the input by as few\n"
    "tokens as possible.\n"
    "\n"
)

In [ ]:
"""
Here is the broken code. Change it to whatever you like.
"""

broken = """
import math

def area(radius)
    return math.pi * radius ** 2

MAX_R = 10
print('calculator ready')

radii = [1, 2, 3 4, 5]
results = {}

for r in radii:
    if r > MAX_R
        print('too big')
        continue
results[r] = area(r)

print('computed', len(results), 'areas')

total = 0
for r, a in results.items():
    total += a
    if a > 50:
    print(f'{r} is large')

average = total / len(results)
print('average:' average)

def report(data, label='summary')
    print(label.upper())
    for k, v in data.items():
        print(f'  {k} -> {v:.2f}')

report(results)
print('finished')
"""

In [ ]:
msgs = [{"role": "user", "content": INSTRUCTION + "\n\n" + broken}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True,
                              return_tensors="pt", return_dict=False).to(model.device)
out = model.generate(ids, max_new_tokens=256, do_sample=False)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


import math

def area(radius):
    return math.pi * radius ** 2

MAX_R = 10
print('calculator ready')

radii = [1, 2, 3, 4, 5]
results = {}

for r in radii:
    if r > MAX_R:
        print('too big')
        continue
    results[r] = area(r)

print('computed', len(results), 'areas')

total = 0
for r, a in results.items():
    total += a
    if a > 50:
        print(f'{r} is large')

average = total / len(results)
print('average:', average)

def report(data, label='summary'):
    print(label.upper())
    for k, v in data.items():
        print(f'  {k} -> {v:.2f}')

report(results)
print('finished')


In [ ]:
import json, difflib
from uuid import uuid4
from IPython.display import HTML, display

def show_diff(original, modified, height=520, side_by_side=True):
    original, modified = original.strip(), modified.strip()

    if original == modified:
        print("No difference — `broken` and `fixed` are identical.")
        return
    n_hunks = sum(1 for op, *_ in difflib.SequenceMatcher(
        None, original.split("\n"), modified.split("\n")).get_opcodes() if op != "equal")
    print(f"{n_hunks} changed region(s)")

    div = f"diff_{uuid4().hex[:8]}"          # unique id: survives cell re-runs
    o, m = json.dumps(original), json.dumps(modified)
    return HTML(f"""
<div id="{div}" style="height:{height}px;border:1px solid #ddd;"></div>
<script src="https://cdn.jsdelivr.net/npm/monaco-editor@0.45.0/min/vs/loader.js"></script>
<script>
  require.config({{ paths: {{ vs: 'https://cdn.jsdelivr.net/npm/monaco-editor@0.45.0/min/vs' }} }});
  require(['vs/editor/editor.main'], function () {{
    const ed = monaco.editor.createDiffEditor(document.getElementById('{div}'), {{
      renderSideBySide: {str(side_by_side).lower()},
      ignoreTrimWhitespace: false,      // indentation fixes MUST stay visible
      readOnly: true,
      automaticLayout: true,
      scrollBeyondLastLine: false,
      renderIndicators: true,
      diffWordWrap: 'on',
      minimap: {{ enabled: false }},
      fontSize: 13
    }});
    ed.setModel({{
      original: monaco.editor.createModel({o}, 'python'),
      modified: monaco.editor.createModel({m}, 'python')
    }});
  }});
</script>
""")

display(show_diff(broken, fixed))

7 changed region(s)
